# KAIROS: velocity-proportional frame and token allocation for long-form action recognition

This notebook reproduces every number, table and figure of the manuscript on a free Google Colab instance.

**Pipeline**
1. write the source files (one `%%writefile` cell per module);
2. download HMDB51 (split 1) and the openly redistributable Something-Something V2 evaluation subset from the Hugging Face Hub, and generate the TRACE-10 benchmark locally;
3. run the camera-compensated pixel probe on every raw frame and encode a uniform candidate lattice with a frozen DINOv2 ViT-S/14 (GPU if available);
4. train all compared methods, ablations and sensitivity runs (three seeds for the main study);
5. aggregate statistics and draw the figures.

**Runtime (free Colab, T4 GPU).** Feature extraction takes roughly 25–40 minutes; the full experiment plan takes roughly 1–2 hours. Set `QUICK = True` below to run one seed of the main comparison only.

`Runtime > Change runtime type > T4 GPU` is recommended but not required.

In [ ]:
QUICK = False          # True: one seed of the main comparison only
N_WORKERS = 2          # parallel training processes

In [ ]:
!pip -q install timm av opencv-python-headless pyarrow scipy nbformat
import torch, platform; print(torch.__version__, platform.python_version(), 'cuda' if torch.cuda.is_available() else 'cpu')

### `common.py`

In [ ]:
%%writefile common.py
"""Shared utilities: frozen DINOv2-S/14 encoder, cheap pixel motion probe, PCA token store."""
import numpy as np, torch, timm, cv2

IMG = 112          # encoder input resolution (8x8 patch grid with patch size 14)
GRID = 8           # patch grid side
NP = GRID * GRID   # patches per frame
PROBE = 56         # probe resolution (7x7 pixel cells)
DPCA = 192         # stored token dimension after PCA
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)


DEV = "cuda" if torch.cuda.is_available() else "cpu"


def load_encoder():
    m = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True,
                          img_size=IMG, num_classes=0).eval().to(DEV)
    for p in m.parameters():
        p.requires_grad_(False)
    return m


def crop_resize(fr, size=IMG):
    """Resize short side to `size`, then centre crop (uint8 RGB in, uint8 out)."""
    h, w = fr.shape[:2]
    s = size / min(h, w)
    nh, nw = max(size, round(h * s)), max(size, round(w * s))
    r = cv2.resize(fr, (nw, nh), interpolation=cv2.INTER_AREA)
    y0, x0 = (nh - size) // 2, (nw - size) // 2
    return r[y0:y0 + size, x0:x0 + size]


def probe_grey(crop112):
    return cv2.resize(cv2.cvtColor(crop112, cv2.COLOR_RGB2GRAY), (PROBE, PROBE), interpolation=cv2.INTER_AREA)


def probe_energy(frames_rgb=None, grey=None):
    """Per-raw-frame, per-cell absolute-difference energy on a 56x56 grey probe.
    Returns e_raw (R, 64) and the camera-compensated residual e_res (R, 64)."""
    if grey is None:
        grey = [probe_grey(crop_resize(f, IMG)) for f in frames_rgb]
    g = np.stack(grey).astype(np.float32)
    d = np.zeros_like(g)
    d[1:] = np.abs(g[1:] - g[:-1])
    c = PROBE // GRID
    e = d.reshape(len(g), GRID, c, GRID, c).mean(axis=(2, 4)).reshape(len(g), NP)
    res = np.maximum(e - np.median(e, axis=1, keepdims=True), 0.0)
    return e, res


def lattice(R, T):
    return np.round(np.linspace(0, R - 1, T)).astype(np.int64)


def aggregate(e, idx):
    """Sum raw-frame energy over the interval (idx[j-1], idx[j]] of each lattice point."""
    cs = np.concatenate([np.zeros((1, e.shape[1]), e.dtype), np.cumsum(e, 0)])
    out = np.zeros((len(idx), e.shape[1]), np.float32)
    prev = idx[0]
    for j, i in enumerate(idx):
        lo = prev + 1 if j > 0 else i
        out[j] = cs[i + 1] - cs[lo] if i >= lo else 0.0
        prev = i
    return out


@torch.inference_mode()
def encode(model, frames112):
    x = (frames112.astype(np.float32) / 255.0 - MEAN) / STD
    x = torch.from_numpy(x).permute(0, 3, 1, 2).contiguous()
    out = []
    for i in range(0, len(x), 64):
        # bfloat16 autocast (AMX on recent Xeon CPUs, tensor cores on GPU); features stay within ~2% of fp32
        with torch.autocast(DEV, dtype=torch.bfloat16):
            out.append(model.forward_features(x[i:i + 64].to(DEV)).float().cpu())  # (b, 65, 384) CLS + 64 patches
    return torch.cat(out).numpy()


class TokenStore:
    """Writes (N, T, 65, DPCA) float16 tokens; PCA is fitted on the first `nfit` training clips."""

    def __init__(self, path, N, T, nfit=120):
        self.mm = np.lib.format.open_memmap(path, mode="w+", dtype=np.float16, shape=(N, T, 1 + NP, DPCA))
        self.buf, self.nfit, self.mu, self.W = [], nfit, None, None

    def _proj(self, f):
        return ((f.reshape(-1, f.shape[-1]) - self.mu) @ self.W).reshape(f.shape[:-1] + (DPCA,)).astype(np.float16)

    def put(self, i, f, is_train):
        if self.W is None:
            self.buf.append((i, f.astype(np.float16), is_train))
            if sum(b[2] for b in self.buf) >= self.nfit:
                self.fit()
            return
        self.mm[i] = self._proj(f)

    def fit(self):
        X = np.concatenate([b[1].reshape(-1, b[1].shape[-1]) for b in self.buf if b[2]]).astype(np.float32)
        rng = np.random.default_rng(0)
        X = X[rng.choice(len(X), min(len(X), 120000), replace=False)]
        self.mu = X.mean(0)
        U, S, Vt = np.linalg.svd(X - self.mu, full_matrices=False)
        self.W = Vt[:DPCA].T.astype(np.float32)
        self.evr = float((S[:DPCA] ** 2).sum() / (S ** 2).sum())
        for i, f, _ in self.buf:
            self.mm[i] = self._proj(f.astype(np.float32))
        self.buf = []

    def close(self):
        if self.W is None:
            self.fit()
        self.mm.flush()


### `synth.py`

In [ ]:
%%writefile synth.py
"""TRACE-10: a controlled long-video benchmark in which the label is carried only by a brief
out-and-back micro-event of one object among look-alike distractors, under camera jitter and
optional camera bumps. Speed classes (fast vs slow jab) differ only in elapsed time."""
import numpy as np, cv2

R, WORLD, VIEW, OFF = 192, 160, 112, 24
CLASSES = ["jab-right-fast", "jab-left-fast", "jab-up-fast", "jab-down-fast", "tilt-cw-fast",
           "tilt-ccw-fast", "pulse-grow-fast", "pulse-shrink-fast", "jab-right-slow", "jab-left-slow"]
SHAPES = [np.array(p, np.float32) for p in [
    [(-1, -.6), (1, 0), (-1, .6), (-.4, 0)],
    [(-1, -1), (-.3, -1), (-.3, .4), (1, .4), (1, 1), (-1, 1)],
    [(-1, -.7), (1, -.7), (1, .7), (.2, .7), (.2, .1), (-.2, .1), (-.2, .7), (-1, .7)],
    [(-1, -1), (-.6, -1), (-.6, -.1), (1, .3), (-.6, .7), (-.6, 1), (-1, 1)]]]


def _bg(rng):
    n = rng.normal(0, 1, (10, 10, 3)).astype(np.float32)
    b = cv2.resize(n, (WORLD, WORLD), interpolation=cv2.INTER_CUBIC)
    b += 0.35 * cv2.resize(rng.normal(0, 1, (40, 40, 3)).astype(np.float32), (WORLD, WORLD), interpolation=cv2.INTER_CUBIC)
    b = (b - b.min()) / (b.max() - b.min() + 1e-6)
    return (70 + 110 * b).astype(np.uint8)


def render(vid, label, seed=12345):
    rng = np.random.default_rng(seed * 100003 + vid)
    bg = _bg(rng)
    nobj = int(rng.integers(2, 4))  # actor + 1..2 distractors
    objs = []
    for k in range(nobj):
        objs.append(dict(shape=SHAPES[int(rng.integers(0, 4))], size=float(rng.uniform(12, 15)),
                         col=tuple(int(c) for c in rng.integers(20, 255, 3)),
                         p=rng.uniform(52, 108, 2), v=rng.uniform(-.25, .25, 2),
                         th=float(rng.uniform(0, 360)), w=float(rng.uniform(-.4, .4))))
    fast = label < 8
    d = int(rng.integers(10, 17)) if fast else int(rng.integers(64, 97))
    t0 = int(rng.integers(12, R - 12 - d))
    # camera: slow random walk + optional bump (global motion burst carrying no label information)
    cam = np.clip(np.cumsum(rng.normal(0, .25, (R, 2)), 0), -6, 6)
    shake = rng.random() < 0.5
    ts = int(rng.integers(4, R - 10)) if shake else -1
    if shake:
        dirv = rng.normal(0, 1, 2); dirv /= np.linalg.norm(dirv) + 1e-6
        for t in range(ts, ts + 6):
            cam[t] += 10 * np.sin(np.pi * (t - ts) / 6) * dirv
    frames = np.empty((R, VIEW, VIEW, 3), np.uint8)
    D = 34.0
    for t in range(R):
        can = bg.copy()
        for k, o in enumerate(objs):
            in_evt = k == 0 and t0 <= t < t0 + d
            if not in_evt:
                o["p"] = o["p"] + o["v"]
                for a in range(2):
                    if o["p"][a] < 52 or o["p"][a] > 108:
                        o["v"][a] *= -1; o["p"][a] = np.clip(o["p"][a], 52, 108)
                o["th"] += o["w"]
            p, th, sc = o["p"].copy(), o["th"], 1.0
            if in_evt:
                u = (t - t0 + 1) / d
                bump = 0.5 * (1 - np.cos(2 * np.pi * u))
                if label in (0, 8): p[0] += D * bump
                elif label in (1, 9): p[0] -= D * bump
                elif label == 2: p[1] -= D * bump
                elif label == 3: p[1] += D * bump
                elif label in (4, 5):
                    th += 90 * bump if label == 4 else -90 * bump
                elif label == 6: sc = 1 + 1.0 * bump
                elif label == 7: sc = 1 - .6 * bump
            c, s_ = np.cos(np.radians(th)), np.sin(np.radians(th))
            pts = o["shape"] @ np.array([[c, s_], [-s_, c]], np.float32) * o["size"] * sc + p
            poly = [np.round(pts * 16).astype(np.int32)]
            cv2.fillPoly(can, poly, o["col"], lineType=cv2.LINE_AA, shift=4)
            if k == 0:   # actor outline
                cv2.polylines(can, poly, True, (15, 15, 15), 2, lineType=cv2.LINE_AA, shift=4)
        oy, ox = int(round(OFF + cam[t, 1])), int(round(OFF + cam[t, 0]))
        frames[t] = can[oy:oy + VIEW, ox:ox + VIEW]
    return frames, dict(t0=t0, t1=t0 + d, shake=ts)


### `prep.py`

In [ ]:
%%writefile prep.py
"""Decode clips, run the pixel motion probe on every raw frame, encode a uniform candidate
lattice with frozen DINOv2-S/14, and store PCA-reduced tokens.
usage: python prep.py {trace|hmdb|ssv2} [--limit N]"""
import sys, os, io, json, time, re, numpy as np, torch
import pyarrow.parquet as pq
from common import *

torch.set_num_threads(2)
name = sys.argv[1]
limit = int(sys.argv[sys.argv.index("--limit") + 1]) if "--limit" in sys.argv else None
LO = int(sys.argv[sys.argv.index("--lo") + 1]) if "--lo" in sys.argv else 0
HI = int(sys.argv[sys.argv.index("--hi") + 1]) if "--hi" in sys.argv else 10 ** 9
out = f"data/{name}"; os.makedirs(out, exist_ok=True)
T = {"trace": 48, "hmdb": 32, "ssv2": 24}[name]
HMDB = ["brush_hair", "cartwheel", "catch", "chew", "clap", "climb", "climb_stairs", "dive", "draw_sword", "dribble",
        "drink", "eat", "fall_floor", "fencing", "flic_flac", "golf", "handstand", "hit", "hug", "jump", "kick",
        "kick_ball", "kiss", "laugh", "pick", "pour", "pullup", "punch", "push", "pushup", "ride_bike", "ride_horse",
        "run", "shake_hands", "shoot_ball", "shoot_bow", "shoot_gun", "sit", "situp", "smile", "smoke", "somersault",
        "stand", "swing_baseball", "sword", "sword_exercise", "talk", "throw", "turn", "walk", "wave"]


def decode(b):
    """Streams the clip once, keeping only 112x112 crops (encoder input) and 56x56 grey probes,
    so that memory stays bounded for very long clips."""
    import av
    crops, grey = [], []
    with av.open(io.BytesIO(b)) as c:
        for f in c.decode(video=0):
            cr = crop_resize(f.to_ndarray(format="rgb24")); crops.append(cr); grey.append(probe_grey(cr))
    return crops, grey


def items():
    """Yields (index, label, split, frames, meta)."""
    if name == "trace":
        from synth import render
        n = [2000, 400, 600]
        lab = np.concatenate([np.random.default_rng(s).permutation(np.arange(k) % 10) for s, k in enumerate(n)])
        spl = np.repeat([0, 1, 2], n)
        for i in range(len(lab)):
            fr, meta = render(i, int(lab[i]))
            yield i, int(lab[i]), int(spl[i]), (list(fr), [probe_grey(f) for f in fr]), meta
        return
    if name == "hmdb":
        files = [("train-0000%d-of-00003" % k, 0) for k in range(3)] + [("test-00000-of-00001", 2)]
        tabs = [(pq.read_table(f"raw/hmdb/{f}.parquet", columns=["label"]), s) for f, s in files]
        paths = []
        for f, s in files:
            pf = pq.ParquetFile(f"raw/hmdb/{f}.parquet")
            for bt in pf.iter_batches(batch_size=16, columns=["video"]):
                paths += [v["path"] for v in bt.column(0).to_pylist()]
        labs = np.concatenate([t[0]["label"].to_numpy() for t in tabs])
        spl = np.concatenate([np.full(len(t), s) for t, s in tabs])
        # group-aware validation split: clips cut from the same source video stay together
        groups = np.array([p.split("_" + HMDB[l] + "_")[0] for p, l in zip(paths, labs)])
        rng = np.random.default_rng(0)
        for c in range(51):
            gs = sorted(set(groups[(labs == c) & (spl == 0)])); rng.shuffle(gs)
            tot, cnt = ((labs == c) & (spl == 0)).sum(), 0
            for g in gs:
                if cnt >= 0.15 * tot: break
                m = (groups == g) & (labs == c) & (spl == 0); spl[m] = 1; cnt += m.sum()
        json.dump(paths, open(f"{out}/paths.json", "w"))
        i = 0
        for f, s in files:
            pf = pq.ParquetFile(f"raw/hmdb/{f}.parquet")
            for bt in pf.iter_batches(batch_size=16, columns=["video"]):   # small batches keep memory bounded
                for v in bt.column(0).to_pylist():
                    if LO <= i < HI:
                        yield i, int(labs[i]), int(spl[i]), decode(v["bytes"]), {}
                    i += 1
                if i >= HI: return
        return
    if name == "ssv2":
        f = "raw/ssv2/test-00000-of-00001.parquet"
        labs = pq.read_table(f, columns=["label"])["label"].to_numpy()
        rng = np.random.default_rng(0); spl = np.zeros(len(labs), np.int64)
        for c in np.unique(labs):
            idx = rng.permutation(np.where(labs == c)[0]); n = len(idx)
            spl[idx[:round(.2 * n)]] = 2; spl[idx[round(.2 * n):round(.3 * n)]] = 1
        pf = pq.ParquetFile(f); i = 0
        for bt in pf.iter_batches(batch_size=16, columns=["video"]):
            for v in bt.column(0).to_pylist():
                yield i, int(labs[i]), int(spl[i]), decode(v["bytes"]), {}
                i += 1


N = {"trace": 3000, "hmdb": 5098, "ssv2": 5444}[name]
if limit: N = limit
enc = load_encoder()
CH = LO > 0 or HI < 10 ** 9
if CH and LO > 0:   # later chunk: reuse PCA and memmap written by the first chunk
    store = TokenStore.__new__(TokenStore)
    store.mm = np.load(f"{out}/tokens.npy", mmap_mode="r+")
    pz = np.load(f"{out}/pca.npz"); store.mu, store.W, store.buf, store.nfit = pz["mu"], pz["W"], [], 0
    store.evr = json.load(open(f"{out}/pca_evr.json"))["evr"]
else:
    store = TokenStore(f"{out}/tokens.npy", N, T)
Eres = np.zeros((N, T, NP), np.float32); Eraw = np.zeros((N, T, NP), np.float32)
labels = np.zeros(N, np.int64); split = np.zeros(N, np.int64); nraw = np.zeros(N, np.int64); metas = []
t0 = time.time()
for i, lab, s, fr, meta in items():
    if i >= N: break
    if not (LO <= i < HI): continue
    crops, grey = fr
    R = len(crops); idx = lattice(R, T)
    e, res = probe_energy(grey=grey)
    Eraw[i], Eres[i] = aggregate(e, idx), aggregate(res, idx)
    cand = np.stack([crops[j] for j in idx])
    store.put(i, encode(enc, cand), s == 0)
    labels[i], split[i], nraw[i] = lab, s, R
    meta.update(lattice=idx.tolist()); metas.append(meta)
    if i % 100 == 0:
        print(f"{name} {i}/{N} {time.time() - t0:.0f}s", flush=True)
store.close()
if CH:   # save this chunk; merge_chunks() assembles the final arrays
    np.savez(f"{out}/chunk_{LO}.npz", lo=LO, hi=min(HI, N), Eres=Eres[LO:HI], Eraw=Eraw[LO:HI], labels=labels[LO:HI],
             split=split[LO:HI], nraw=nraw[LO:HI], metas=json.dumps(metas))
    if LO == 0:
        np.savez(f"{out}/pca.npz", mu=store.mu, W=store.W); json.dump(dict(evr=store.evr), open(f"{out}/pca_evr.json", "w"))
    print("chunk done", LO, HI, time.time() - t0); sys.exit(0)
np.save(f"{out}/Eres.npy", Eres); np.save(f"{out}/Eraw.npy", Eraw)
np.save(f"{out}/labels.npy", labels); np.save(f"{out}/split.npy", split); np.save(f"{out}/nraw.npy", nraw)
np.savez(f"{out}/pca.npz", mu=store.mu, W=store.W)
json.dump(dict(T=T, N=N, pca_evr=store.evr, seconds=time.time() - t0, metas=metas), open(f"{out}/meta.json", "w"))
print("done", name, time.time() - t0, "evr", store.evr)


### `kairos.py`

In [ ]:
%%writefile kairos.py
"""Samplers, token reducers, head network and training loop for velocity-proportional
spatiotemporal token allocation on frozen-encoder tokens."""
import math, json, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
GRID = 4          # head grid after 2x2 pooling of the 8x8 DINOv2 patch grid
NP = GRID * GRID


def pool_tokens(x):
    """(..., 65, D) -> (..., 1+NP, D): CLS kept, 8x8 patches average-pooled to GRIDxGRID."""
    if GRID == 8: return x
    c, p = x[..., :1, :], x[..., 1:, :]
    sh = p.shape[:-2]; f = 8 // GRID
    p = p.reshape(sh + (GRID, f, GRID, f, p.shape[-1])).mean((-4, -2)).reshape(sh + (NP, p.shape[-1]))
    return np.concatenate([c, p], -2)


def pool_motion(m):
    if GRID == 8: return m
    sh = m.shape[:-1]; f = 8 // GRID
    return m.reshape(sh + (GRID, f, GRID, f)).sum((-3, -1)).reshape(sh + (NP,))


# ----------------------------------------------------------------------------- data
def pool_file(root, chunk=200):
    """Writes tokens_p.npy: CLS + GRIDxGRID pooled patch tokens (float16), loaded fully into memory later."""
    src = np.load(f"{root}/tokens.npy", mmap_mode="r")
    out = np.lib.format.open_memmap(f"{root}/tokens_p.npy", mode="w+", dtype=np.float16, shape=src.shape[:2] + (1 + NP, src.shape[-1]))
    for i in range(0, len(src), chunk):
        out[i:i + chunk] = pool_tokens(np.asarray(src[i:i + chunk], np.float32)).astype(np.float16)
    out.flush()


class Bank:
    def __init__(self, root, light=False):
        import os
        if not os.path.exists(f"{root}/tokens_p.npy"):
            pool_file(root)
        self.tok = np.load(f"{root}/tokens_p.npy", mmap_mode="r" if light else None)
        self.Eres = pool_motion(np.load(f"{root}/Eres.npy")); self.Eraw = pool_motion(np.load(f"{root}/Eraw.npy"))
        self.Etm = np.maximum(self.Eres - np.median(self.Eres, 1, keepdims=True), 0)
        self.y = np.load(f"{root}/labels.npy"); self.split = np.load(f"{root}/split.npy")
        self.N, self.T = self.tok.shape[:2]; self.D = self.tok.shape[-1]
        self.C = int(self.y.max()) + 1
        self.root = root

    def ids(self, s):
        return np.where(self.split == s)[0]


# ----------------------------------------------------------------------------- samplers
def _strict(idx, T):
    idx = np.asarray(idx, np.int64).copy(); K = len(idx)
    for k in range(1, K):
        idx[k] = max(idx[k], idx[k - 1] + 1)
    for k in range(K - 1, -1, -1):
        idx[k] = min(idx[k], T - K + k)
        if k < K - 1: idx[k] = min(idx[k], idx[k + 1] - 1)
    return idx


def quantile_sample(v, K, rho, gamma, rng=None):
    """Velocity-quantile sampling: density (1-rho) v^gamma / sum + rho / T on the lattice."""
    T = len(v); w = np.power(np.maximum(v, 0) + 1e-6, gamma); w = w / w.sum()
    p = (1 - rho) * w + rho / T
    Fc = np.cumsum(p); Fc[-1] = 1.0
    u = rng.random(K) if rng is not None else np.full(K, 0.5)
    q = (np.arange(K) + u) / K
    return _strict(np.searchsorted(Fc, q, side="left"), T)


def uniform_sample(T, K, rng=None):
    edges = np.linspace(0, T, K + 1)
    if rng is None:
        idx = np.floor((edges[:-1] + edges[1:]) / 2)
    else:
        idx = np.floor(edges[:-1] + rng.random(K) * (edges[1:] - edges[:-1]))
    return _strict(np.clip(idx, 0, T - 1), T)


def tmed(E):
    """Remove the persistent (temporal-median) motion level of every region: (T, N) -> (T, N)."""
    return np.maximum(E - np.median(E, 0, keepdims=True), 0)


def select(bank, i, K, sampler, rng=None, Eres=None, Eraw=None, T=None):
    Eres = bank.Eres[i] if Eres is None else Eres
    Eraw = bank.Eraw[i] if Eraw is None else Eraw
    T = Eres.shape[0]
    if sampler["name"] == "uniform":
        return uniform_sample(T, K, rng)
    if sampler["name"] == "mg":      # cumulative raw motion, no compensation, no coverage floor
        return quantile_sample(Eraw.sum(1), K, 0.0, 1.0, rng)
    if sampler["name"] == "topk":    # K most active lattice points
        v = Eraw.sum(1) + (rng.random(T) * 1e-3 if rng is not None else 0)
        return np.sort(np.argsort(-v)[:K])
    if sampler["name"] == "vqs":
        E = tmed(Eres if sampler.get("comp", True) else Eraw)
        return quantile_sample(E.sum(1), K, sampler.get("rho", .2), sampler.get("gamma", 2.0), rng)
    raise ValueError(sampler)


def make_batch(bank, ids, K, sampler, rng=None, view=None):
    """Returns tokens (B,K,65,D), motion (B,K,64), tau (B,K), labels.
    `view(i)` may return a transformed clip (tokens, Eres, Eraw, tau_lattice) for stress protocols."""
    X, M, TAU = [], [], []
    for i in ids:
        if view is None:
            idx = select(bank, i, K, sampler, rng)
            X.append(bank.tok[i, idx].astype(np.float32)); M.append(bank.Etm[i, idx])
            TAU.append(idx / (bank.T - 1))
        else:
            tok, Er, Ew, tl = view(i)
            idx = select(bank, i, K, sampler, rng, Er, Ew)
            X.append(np.asarray(tok[idx], np.float32)); M.append(tmed(Er)[idx]); TAU.append(tl[idx])
    return (torch.from_numpy(np.stack(X)), torch.from_numpy(np.stack(M)).float(),
            torch.from_numpy(np.stack(TAU)).float(), torch.from_numpy(bank.y[ids]))


# ----------------------------------------------------------------------------- model
def fourier(x, nf=6):
    f = (2.0 ** torch.arange(nf, device=x.device, dtype=x.dtype)) * math.pi
    a = x[..., None] * f
    return torch.cat([a.sin(), a.cos()], -1)


class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__()
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.o = nn.Linear(d, 3 * d), nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(drop), nn.Linear(2 * d, d))
        self.h, self.drop = h, drop

    def forward(self, x, bias):
        B, L, d = x.shape
        q, k, v = self.qkv(self.n1(x)).view(B, L, 3, self.h, d // self.h).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, attn_mask=bias, dropout_p=self.drop if self.training else 0.)
        x = x + self.o(a.transpose(1, 2).reshape(B, L, d))
        return x + self.mlp(self.n2(x))


class Head(nn.Module):
    """cfg keys: reducer in {full, cls, random, stts, tome, evit, rlt, ours}; budget B (patch tokens);
    enc in {index, ct} ; ctx (context tokens, ours) ; prior (motion+novelty prior, ours) ; glob (global budget)."""

    def __init__(self, D, C, cfg, d=128, L=2, h=4, drop=0.1, Kmax=64):
        super().__init__()
        self.cfg = cfg
        self.inp = nn.Linear(D, d)
        self.pos = nn.Parameter(torch.zeros(NP + 2, d)); nn.init.trunc_normal_(self.pos, std=.02)
        self.read = nn.Parameter(torch.zeros(1, 1, d))
        self.slot = nn.Embedding(Kmax, d)
        self.tenc, self.genc = nn.Linear(12, d), nn.Linear(12, d)
        self.vel = nn.Linear(D, d)
        self.g_geo, self.g_vel = nn.Parameter(torch.zeros(1)), nn.Parameter(torch.zeros(1))
        self.scorer = nn.Sequential(nn.Linear(D, 64), nn.GELU(), nn.Linear(64, 1))
        self.wm, self.wn = nn.Parameter(torch.tensor(1.0)), nn.Parameter(torch.tensor(1.0))
        self.blocks = nn.ModuleList([Block(d, h, drop) for _ in range(L)])
        self.norm, self.fc = nn.LayerNorm(d), nn.Linear(d, C)

    # ---- temporal encodings
    def temporal(self, tau, Kn):
        B, K = tau.shape
        enc = self.cfg.get("enc", "index")
        if enc == "index":
            return self.slot(torch.arange(K, device=tau.device))[None].expand(B, K, -1)
        gap = torch.cat([tau[:, :1] * 0 + 1.0 / max(Kn, 1), tau[:, 1:] - tau[:, :-1]], 1).clamp_min(1e-3)
        lg = torch.log(gap * Kn) / 4.0                       # 0 when spacing equals the uniform spacing
        if enc == "ctg":                                     # order embedding + zero-initialised gated geometry terms
            return (self.slot(torch.arange(K, device=tau.device))[None].expand(B, K, -1)
                    + self.g_geo * (self.tenc(fourier(tau)) + self.genc(fourier(lg))))
        if enc == "rel":                                     # order + intervals only (invariant to time translation)
            return self.slot(torch.arange(K, device=tau.device))[None].expand(B, K, -1) + self.genc(fourier(lg))
        return self.tenc(fourier(tau)) + self.genc(fourier(lg))

    def forward(self, X, M, tau):
        cfg = self.cfg; B, K, _, D = X.shape; dv = X.device
        cls_t, P = X[:, :, 0], X[:, :, 1:]                    # (B,K,D), (B,K,64,D)
        Prev = torch.cat([P[:, :1], P[:, :-1]], 1)
        te = self.temporal(tau, K)                            # (B,K,d)
        use_vel = cfg.get("enc", "index") in ("ct", "rel", "ctg") and cfg.get("vel", True)
        gv = self.g_vel if cfg.get("enc") == "ctg" else 1.0
        if use_vel:
            gap = torch.cat([torch.ones_like(tau[:, :1]), (tau[:, 1:] - tau[:, :-1]) * K], 1).clamp_min(.05)
            Vp = (P - Prev) / gap[..., None, None]
            Vc = (cls_t - torch.cat([cls_t[:, :1], cls_t[:, :-1]], 1)) / gap[..., None]
        red = cfg["reducer"]
        toks, bias = [self.read.expand(B, -1, -1)], [torch.zeros(B, 1, device=dv)]
        ec = self.inp(cls_t) + self.pos[0] + te + (gv * self.vel(Vc) if use_vel else 0)
        toks.append(ec); bias.append(torch.zeros(B, K, device=dv))
        if red == "cls":
            pass
        elif red == "full":
            e = self.inp(P) + self.pos[1:NP + 1] + te[:, :, None] + (gv * self.vel(Vp) if use_vel else 0)
            toks.append(e.reshape(B, K * NP, -1)); bias.append(torch.zeros(B, K * NP, device=dv))
        else:
            b = cfg["B"]; per = b // K
            if red == "tome":
                kept, mass, pid = self._tome(P, per)
                e = self.inp(kept) + self.pos[1:NP + 1][pid] + te[:, :, None]
                toks.append(e.reshape(B, -1, e.shape[-1])); bias.append(torch.log(mass).reshape(B, -1))
            else:
                s, extra = self._scores(P, Prev, M, cls_t, red)
                if red in ("ours",) and cfg.get("glob", True) or red == "rlt":
                    nb = b
                    flat = s.reshape(B, K * NP)
                    if red == "rlt":
                        flat = flat.clone(); flat[:, :NP] = 1e9   # first frame kept in full
                    top = flat.topk(nb, 1).indices
                    keep = torch.zeros(B, K * NP, dtype=torch.bool, device=dv).scatter_(1, top, True).view(B, K, NP)
                else:
                    top = s.topk(per, -1).indices
                    keep = torch.zeros(B, K, NP, dtype=torch.bool, device=dv).scatter_(2, top, True)
                kf = keep.reshape(B, -1); order = torch.sort(kf.float(), stable=True, dim=1, descending=True).indices[:, :int(kf.sum(1).min())]
                Pf = P.reshape(B, K * NP, D).gather(1, order[..., None].expand(-1, -1, D))
                kidx, nidx = order // NP, order % NP
                e = self.inp(Pf) + self.pos[1:NP + 1][nidx] + te.gather(1, kidx[..., None].expand(-1, -1, te.shape[-1]))
                if use_vel:
                    e = e + gv * self.vel(Vp.reshape(B, K * NP, D).gather(1, order[..., None].expand(-1, -1, D)))
                if red in ("stts", "ours"):              # straight-through: forward value 1, gradient to the scorer
                    g = torch.sigmoid(s.reshape(B, -1).gather(1, order))
                    e = e * (1 + g - g.detach())[..., None]
                toks.append(e); bias.append(torch.zeros(B, e.shape[1], device=dv))
                if red == "evit" or (red == "ours" and cfg.get("ctx", True)):
                    drop = (~keep).float()
                    cnt = drop.sum(-1)
                    ctx = (P * drop[..., None]).sum(2) / cnt.clamp_min(1)[..., None]
                    ec2 = self.inp(ctx) + self.pos[NP + 1] + te
                    toks.append(ec2); bias.append(torch.log(cnt.clamp_min(1e-4)))
        x = torch.cat(toks, 1)
        bb = torch.cat(bias, 1)[:, None, None, :].to(x.dtype)
        for blk in self.blocks:
            x = blk(x, bb)
        return self.fc(self.norm(x[:, 0]))

    def _scores(self, P, Prev, M, cls_t, red):
        B, K = P.shape[:2]
        if red == "random":
            return torch.rand(B, K, NP, device=P.device), None
        if red == "evit":
            return F.cosine_similarity(P, cls_t[:, :, None], dim=-1), None
        nov = 1 - F.cosine_similarity(P, Prev, dim=-1)       # first frame -> 0
        if red == "rlt":
            return nov, None
        learned = self.scorer(P).squeeze(-1)
        if red == "stts" or not self.cfg.get("prior", True):
            return learned, None

        def z(a):
            mu = a.mean((1, 2), keepdim=True); sd = a.std((1, 2), keepdim=True) + 1e-6
            return (a - mu) / sd
        return self.wm * z(torch.log1p(M)) + self.wn * z(nov) + learned, None

    @staticmethod
    def _tome(P, per):
        """Bipartite soft matching (training-free) per frame, applied until `per` tokens remain."""
        B, K, N, D = P.shape
        x = P.reshape(B * K, N, D); size = torch.ones(B * K, N, device=P.device); pid = torch.arange(N, device=P.device).expand(B * K, N).clone()
        while x.shape[1] > per:
            n = x.shape[1]; r = min(n // 2, n - per)
            a, bt = x[:, ::2], x[:, 1::2]; sa, sb = size[:, ::2], size[:, 1::2]
            sim = F.normalize(a, dim=-1) @ F.normalize(bt, dim=-1).transpose(1, 2)
            best, j = sim.max(-1)
            order = best.argsort(-1, descending=True)
            src, keep_a = order[:, :r], order[:, r:]
            dst = j.gather(1, src)
            ws = sa.gather(1, src)
            xb = bt * sb[..., None]
            xb = xb.scatter_add(1, dst[..., None].expand(-1, -1, D), a.gather(1, src[..., None].expand(-1, -1, D)) * ws[..., None])
            sb2 = sb.scatter_add(1, dst, ws)
            xb = xb / sb2[..., None]
            x = torch.cat([a.gather(1, keep_a[..., None].expand(-1, -1, D)), xb], 1)
            size = torch.cat([sa.gather(1, keep_a), sb2], 1)
            pid = torch.cat([pid[:, ::2].gather(1, keep_a), pid[:, 1::2]], 1)
        return x.view(B, K, per, D), size.view(B, K, per), pid.view(B, K, per)


class MeanCLS(nn.Module):
    """TSN-style consensus: average of frame CLS features followed by a linear classifier."""

    def __init__(self, D, C, cfg=None):
        super().__init__(); self.n = nn.LayerNorm(D); self.fc = nn.Linear(D, C); self.cfg = cfg or {}

    def forward(self, X, M, tau):
        return self.fc(self.n(X[:, :, 0].mean(1)))


# ----------------------------------------------------------------------------- train / eval
def build(bank, cfg):
    if cfg["reducer"] == "mean":
        return MeanCLS(bank.D, bank.C, cfg)
    return Head(bank.D, bank.C, cfg)


@torch.no_grad()
def predict(model, bank, ids, K, sampler, bs=128, view=None):
    model.eval(); out = []
    for j in range(0, len(ids), bs):
        X, M, tau, _ = make_batch(bank, ids[j:j + bs], K, sampler, None, view)
        out.append(F.softmax(model(X.to(DEV), M.to(DEV), tau.to(DEV)), -1).cpu())
    return torch.cat(out).numpy()


def train(bank, cfg, seed=0, epochs=20, K=8, bs=64, lr=1e-3, log=None):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = build(bank, cfg).to(DEV)
    sampler = cfg["sampler"]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    tr, va = bank.ids(0), bank.ids(1)
    steps = epochs * math.ceil(len(tr) / bs); warm = math.ceil(len(tr) / bs)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    best, best_state, t0 = -1, None, time.time()
    for ep in range(epochs):
        model.train(); perm = rng.permutation(tr)
        for j in range(0, len(perm), bs):
            X, M, tau, y = make_batch(bank, perm[j:j + bs], K, sampler, rng)
            loss = F.cross_entropy(model(X.to(DEV), M.to(DEV), tau.to(DEV)), y.to(DEV), label_smoothing=0.1)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        acc = (predict(model, bank, va, K, sampler).argmax(1) == bank.y[va]).mean()
        if acc > best:
            best, best_state = acc, {k: v.detach().clone() for k, v in model.state_dict().items()}
        if log: log(f"ep{ep} val {acc:.4f} {time.time() - t0:.0f}s")
    model.load_state_dict(best_state)
    return model, best, time.time() - t0


### `plans.py`

In [ ]:
%%writefile plans.py
"""Method definitions and experiment plans."""
from kairos import NP

U = dict(name="uniform"); MG = dict(name="mg"); TOPK = dict(name="topk")


def VQS(rho=.2, gamma=2.0, comp=True):
    return dict(name="vqs", rho=rho, gamma=gamma, comp=comp)


def budget(red, K=8, beta=.25):
    b = int(round(beta * K * NP))
    return b - K if red in ("evit", "ours") else b


def m(sampler, red, K=8, beta=.25, **kw):
    c = dict(sampler=sampler, reducer=red, K=K, **kw)
    if red not in ("mean", "cls", "full"):
        c["B"] = budget(red, K, beta)
    return c


OURS = dict(enc="ctg", ctx=True, prior=True, glob=True, vel=True)
METHODS = {
    "TSN-mean": m(U, "mean"),
    "Uniform-CLS": m(U, "cls"),
    "Dense-CLS": m(U, "cls", K="T"),
    "Uniform-Full": m(U, "full"),
    "MG-Full": m(MG, "full"),
    "Uniform-Random": m(U, "random"),
    "Uniform-STTS": m(U, "stts"),
    "Uniform-ToMe": m(U, "tome"),
    "Uniform-EViT": m(U, "evit"),
    "Uniform-RLT": m(U, "rlt"),
    "MG-STTS": m(MG, "stts"),
    "Ours": m(VQS(), "ours", **OURS),
}
# factorial ablation over A (sampler), B (cross-frame velocity allocation), C (sampling-geometry encoding)
for a in (0, 1):
    for bb in (0, 1):
        for c in (0, 1):
            nm = f"Abl-A{a}B{bb}C{c}"
            METHODS[nm] = m(VQS() if a else U, "ours", enc="ctg" if c else "index", ctx=True,
                            prior=bool(bb), glob=bool(bb), vel=True)
# component-level removals
METHODS["Ours-noComp"] = m(VQS(comp=False), "ours", **OURS)
METHODS["Ours-rho0"] = m(VQS(rho=0.0), "ours", **OURS)
METHODS["Ours-noCtx"] = m(VQS(), "ours", **{**OURS, "ctx": False})
METHODS["Ours-noVel"] = m(VQS(), "ours", **{**OURS, "vel": False})
METHODS["Ours-ungated"] = m(VQS(), "ours", **{**OURS, "enc": "ct"})
# sampler swap under the full proposed head
METHODS["Samp-Uniform"] = m(U, "ours", **OURS)
METHODS["Samp-TopK"] = m(TOPK, "ours", **OURS)
METHODS["Samp-MG"] = m(MG, "ours", **OURS)
# sensitivity
for r in (0.05, 0.1, 0.4, 0.7, 1.0):
    METHODS[f"Sens-rho{r}"] = m(VQS(rho=r), "ours", **OURS)
for g in (1.0, 3.0):
    METHODS[f"Sens-gamma{g}"] = m(VQS(gamma=g), "ours", **OURS)
for be in (0.125, 0.5, 1.0):
    METHODS[f"Sens-beta{be}"] = m(VQS(), "ours", beta=be, **OURS)
    METHODS[f"Sens-STTS-beta{be}"] = m(U, "stts", beta=be)
for k in (4, 16):
    METHODS[f"Sens-K{k}"] = m(VQS(), "ours", K=k, **OURS)
    METHODS[f"Sens-U-Full-K{k}"] = m(U, "full", K=k)

DS = ["trace", "hmdb", "ssv2"]
STRESS = [("dilution", .5), ("dilution", .25), ("dilution", .125), ("noise", .5), ("noise", 1.0),
          ("K", 4), ("K", 12), ("K", 16)]
MAIN = ["TSN-mean", "Uniform-CLS", "Dense-CLS", "Uniform-Full", "MG-Full", "Uniform-Random", "Uniform-STTS",
        "Uniform-ToMe", "Uniform-EViT", "Uniform-RLT", "MG-STTS", "Ours"]
ABL = [f"Abl-A{a}B{b}C{c}" for a in (0, 1) for b in (0, 1) for c in (0, 1)]
COMP = ["Ours-noComp", "Ours-rho0", "Ours-noCtx", "Ours-noVel", "Ours-ungated", "Samp-TopK", "Samp-MG"]
SENS = [k for k in METHODS if k.startswith("Sens-")]


def _jobs(names, dss, seeds, extra=False, epochs=20):
    J = []
    for ds in dss:
        for s in seeds:
            for n in names:
                if n == "Abl-A1B1C1":   # identical to Ours
                    continue
                j = dict(ds=ds, name=n, seed=s, cfg=METHODS[n], epochs=epochs)
                if extra and n in MAIN: j["extra"] = STRESS
                if n in ("Ours", "Uniform-STTS") and s == 0: j["save_model"] = True
                J.append(j)
    return J


PLANS = {
    "pilot": lambda: _jobs(["Uniform-CLS", "Uniform-STTS", "Ours"], ["trace"], [0], epochs=20),
    "main": lambda: _jobs(MAIN, DS, [0, 1, 2], extra=True),
    "abl": lambda: _jobs(ABL, DS, [0, 1, 2]) + _jobs(COMP, ["trace", "hmdb"], [0, 1]) + _jobs(COMP, ["ssv2"], [0]),
    "sens": lambda: _jobs(SENS, ["trace", "hmdb"], [0]),
}


### `run_exp.py`

In [ ]:
%%writefile run_exp.py
"""Job runner. usage: python run_exp.py <plan> <worker_id> <n_workers>
Each job trains one configuration on one dataset/seed and stores validation accuracy,
test probabilities and stress-protocol probabilities under results/."""
import sys, os, json, time, numpy as np, torch
torch.set_num_threads(1)
import kairos as kz
from plans import PLANS, METHODS

plan, wid, nw = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
plan, _, only = plan.partition(":")
jobs = [j for j in PLANS[plan]() if not only or j["ds"] == only]
BANKS = {}


def bank(ds):
    if ds not in BANKS:
        BANKS[ds] = kz.Bank(f"data/{ds}")
    return BANKS[ds]


def dilution_view(b, phi, seed=0):
    """Idle-stream protocol: the clip occupies a fraction phi of the lattice and the remaining
    slots repeat its first/last frame with sensor-level probe noise."""
    T = b.T
    def view(i):
        rng = np.random.default_rng(seed * 7919 + int(i))
        n = max(2, int(round(T * phi)))
        src = np.round(np.linspace(0, T - 1, n)).astype(int)
        off = int(rng.integers(0, T - n + 1))
        slot = np.empty(T, int); slot[:off] = src[0]; slot[off:off + n] = src; slot[off + n:] = src[-1]
        tok = b.tok[i][slot]
        Er = np.zeros((T, b.Eres.shape[2]), np.float32); Ew = np.zeros_like(Er)
        cr, cw = np.cumsum(b.Eres[i], 0), np.cumsum(b.Eraw[i], 0)
        for j in range(n):
            lo = src[j - 1] if j > 0 else src[0] - 1
            Er[off + j] = cr[src[j]] - (cr[lo] if lo >= 0 else 0)
            Ew[off + j] = cw[src[j]] - (cw[lo] if lo >= 0 else 0)
        s0 = np.percentile(b.Eraw[i], 5) + 1e-3
        mask = np.ones(T, bool); mask[off:off + n] = False
        Ew[mask] = np.abs(rng.normal(0, s0, (mask.sum(), Ew.shape[1])))
        Er[mask] = np.maximum(Ew[mask] - np.median(Ew[mask], 1, keepdims=True), 0)
        return tok, Er, Ew, np.arange(T) / (T - 1)
    return view


def noise_view(b, sigma, seed=0):
    def view(i):
        rng = np.random.default_rng(seed * 104729 + int(i))
        s = sigma * b.Eraw[i].mean()
        Ew = np.abs(b.Eraw[i] + rng.normal(0, s, b.Eraw[i].shape)).astype(np.float32)
        Er = np.maximum(b.Eres[i] + rng.normal(0, s, b.Eres[i].shape), 0).astype(np.float32)
        return b.tok[i], Er, Ew, np.arange(b.T) / (b.T - 1)
    return view


def interp_slots(model, K_new, K_train):
    """Linear interpolation of learned slot embeddings for index-encoded heads evaluated at another K."""
    if not hasattr(model, "slot") or model.cfg.get("enc", "index") != "index" or K_new == K_train:
        return None
    W = model.slot.weight.data[:K_train].clone()
    pos = torch.linspace(0, K_train - 1, K_new)
    lo = pos.floor().long().clamp(max=K_train - 1); hi = (lo + 1).clamp(max=K_train - 1); a = (pos - lo)[:, None]
    old = model.slot.weight.data.clone()
    model.slot.weight.data[:K_new] = (1 - a) * W[lo] + a * W[hi]
    return old


for n, job in enumerate(jobs):
    if n % nw != wid:
        continue
    out = f"results/{job['ds']}/{job['name']}/seed{job['seed']}"
    if os.path.exists(out + "/done.json"):
        continue
    os.makedirs(out, exist_ok=True)
    try:                                  # lock so that several workers can share one plan
        fd = os.open(out + "/lock", os.O_CREAT | os.O_EXCL); os.close(fd)
    except FileExistsError:
        continue
    b = bank(job["ds"]); cfg = dict(job["cfg"])
    K = cfg.pop("K", 8)
    if K == "T": K = b.T
    logf = open(out + "/log.txt", "w")
    model, val, secs = kz.train(b, cfg, seed=job["seed"], epochs=job.get("epochs", 20), K=K,
                                log=lambda s: (logf.write(s + "\n"), logf.flush()))
    te = b.ids(2)
    t0 = time.time(); P = kz.predict(model, b, te, K, cfg["sampler"]); inf_s = time.time() - t0
    np.save(out + "/test_probs.npy", P.astype(np.float16))
    acc = float((P.argmax(1) == b.y[te]).mean())
    res = dict(job=job, K=K, val=float(val), test=acc, train_s=secs, infer_s=inf_s,
               params=int(sum(p.numel() for p in model.parameters())))
    for ex in job.get("extra", []):
        if ex[0] == "dilution":
            P2 = kz.predict(model, b, te, K, cfg["sampler"], view=dilution_view(b, ex[1]))
            np.save(out + f"/dil{ex[1]}.npy", P2.astype(np.float16)); res[f"dil{ex[1]}"] = float((P2.argmax(1) == b.y[te]).mean())
        elif ex[0] == "noise":
            P2 = kz.predict(model, b, te, K, cfg["sampler"], view=noise_view(b, ex[1]))
            res[f"noise{ex[1]}"] = float((P2.argmax(1) == b.y[te]).mean())
        elif ex[0] == "K":
            old = interp_slots(model, ex[1], K)
            P2 = kz.predict(model, b, te, ex[1], cfg["sampler"])
            if old is not None: model.slot.weight.data.copy_(old)
            res[f"K{ex[1]}"] = float((P2.argmax(1) == b.y[te]).mean())
    if job.get("save_model"):
        torch.save(model.state_dict(), out + "/model.pt")
    json.dump(res, open(out + "/done.json", "w"), indent=1)
    print(job["ds"], job["name"], job["seed"], f"val {val:.4f} test {acc:.4f} {secs:.0f}s", flush=True)


### `analyze.py`

In [ ]:
%%writefile analyze.py
"""Aggregate results/ into summary.json used by tables, figures and text."""
import json, os, glob, numpy as np
from scipy import stats
import kairos as kz
from plans import METHODS, MAIN, ABL, SENS, DS

rng = np.random.default_rng(0)
S = {"runs": {}, "hit": {}, "stats": {}, "data": {}}


def ece(P, y, Q=15):
    c = P.max(1); a = (P.argmax(1) == y); e = 0
    edges = np.linspace(0, 1, Q + 1)
    for q in range(Q):
        m = (c > edges[q]) & (c <= edges[q + 1])
        if m.any(): e += m.mean() * abs(a[m].mean() - c[m].mean())
    return float(e)


def macro_f1(pred, y, C):
    f = []
    for c in range(C):
        tp = ((pred == c) & (y == c)).sum(); fp = ((pred == c) & (y != c)).sum(); fn = ((pred != c) & (y == c)).sum()
        f.append(2 * tp / max(2 * tp + fp + fn, 1))
    return float(np.mean(f))


banks = {ds: kz.Bank(f"data/{ds}", light=True) for ds in DS if os.path.exists(f"data/{ds}/labels.npy")}
for ds, b in banks.items():
    y = b.y[b.ids(2)]
    S["data"][ds] = dict(N=int(b.N), T=int(b.T), C=int(b.C), tr=int((b.split == 0).sum()), va=int((b.split == 1).sum()),
                         te=int((b.split == 2).sum()), nraw_mean=float(np.load(f"data/{ds}/nraw.npy").mean()),
                         pca_evr=json.load(open(f"data/{ds}/meta.json"))["pca_evr"])
    for d in sorted(glob.glob(f"results/{ds}/*")):
        name = os.path.basename(d); R = []
        for sd in sorted(glob.glob(d + "/seed*/done.json")):
            r = json.load(open(sd)); P = np.load(os.path.dirname(sd) + "/test_probs.npy").astype(np.float32)
            r["f1"] = macro_f1(P.argmax(1), y, b.C); r["ece"] = ece(P, y)
            r["nll"] = float(-np.log(P[np.arange(len(y)), y] + 1e-8).mean())
            r["top5"] = float((np.argsort(-P, 1)[:, :5] == y[:, None]).any(1).mean())
            r["correct"] = (P.argmax(1) == y).astype(np.int8).tolist()
            R.append(r)
        if not R: continue
        agg = {}
        for k in R[0]:
            if isinstance(R[0][k], float) and all(k in r for r in R):
                v = np.array([r[k] for r in R]); agg[k] = [float(v.mean()), float(v.std(ddof=0)), len(v)]
        agg["correct_mean"] = np.mean([r["correct"] for r in R], 0).tolist()
        agg["correct_seeds"] = [r["correct"] for r in R]
        S["runs"].setdefault(ds, {})[name] = agg

# --------------------------------------------------------- burst hit rate of samplers (TRACE-10 test)
if "trace" in banks:
    b = banks["trace"]; meta = json.load(open("data/trace/meta.json"))["metas"]
    te = b.ids(2); lat_step = 4
    for sname, smp in [("Uniform", dict(name="uniform")), ("Top-K motion", dict(name="topk")), ("MGSampler", dict(name="mg")),
                       ("VQS, no compensation", dict(name="vqs", comp=False)), ("VQS, rho=0", dict(name="vqs", rho=0.0)), ("VQS, gamma=1", dict(name="vqs", gamma=1.0)),
                       ("VQS (KAIROS)", dict(name="vqs"))]:
        H, two, shake_hit, H_shk, H_noshk = [], [], [], [], []
        for i in te:
            m = meta[i]; lat = np.array(m["lattice"]); idx = kz.select(b, i, 8, smp)
            r = lat[idx]
            inside = (r >= m["t0"]) & (r < m["t1"] + lat_step)
            H.append(inside.mean()); two.append(inside.sum() >= 2)
            if m["shake"] >= 0:
                shake_hit.append(((r >= m["shake"]) & (r < m["shake"] + 6 + lat_step)).mean()); H_shk.append(inside.mean())
            else:
                H_noshk.append(inside.mean())
        S["hit"][sname] = dict(H=float(np.mean(H)), two=float(np.mean(two)), shake=float(np.mean(shake_hit)),
                               H_shake=float(np.mean(H_shk)), H_noshake=float(np.mean(H_noshk)))
    # fast vs slow classes / shake vs no-shake breakdown for main methods
    y = b.y[te]; shk = np.array([meta[i]["shake"] >= 0 for i in te])
    for name, agg in S["runs"].get("trace", {}).items():
        c = np.array(agg["correct_mean"])
        agg["acc_shake"] = float(c[shk].mean()); agg["acc_noshake"] = float(c[~shk].mean())
        agg["acc_slow"] = float(c[y >= 8].mean()); agg["acc_fastjab"] = float(c[(y <= 1)].mean())
        agg["per_class"] = [float(c[y == k].mean()) for k in range(10)]

# --------------------------------------------------------- statistics: Ours vs baselines
def boot(a, b_, n=10000):
    d = a - b_; idx = rng.integers(0, len(d), (n, len(d)))
    bs = d[idx].mean(1)
    p = 2 * min((bs <= 0).mean(), (bs >= 0).mean())          # two-sided bootstrap p-value
    return float(d.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)), float(min(1.0, max(p, 1.0 / n)))


def mcnemar(a, b_):
    n01 = int(((a == 1) & (b_ == 0)).sum()); n10 = int(((a == 0) & (b_ == 1)).sum())
    if n01 + n10 == 0: return 1.0
    return float(stats.binomtest(n01, n01 + n10, 0.5).pvalue)


for ds in S["runs"]:
    if "Ours" not in S["runs"][ds]: continue
    o = np.array(S["runs"][ds]["Ours"]["correct_mean"]); ov = (o > 0.5).astype(int)
    res = {}
    for name in MAIN:
        if name == "Ours" or name not in S["runs"][ds]: continue
        c = np.array(S["runs"][ds][name]["correct_mean"])
        d, lo, hi, p = boot(o, c); res[name] = dict(delta=d, lo=lo, hi=hi, p=p, mcnemar=mcnemar(ov, (c > 0.5).astype(int)))
    # Holm
    names = list(res); ps = np.array([res[n]["p"] for n in names]); order = np.argsort(ps); m = len(ps); adj = np.zeros(m)
    run = 0
    for r_, i in enumerate(order):
        run = max(run, min(1, (m - r_) * ps[i])); adj[i] = run
    for n, a in zip(names, adj): res[n]["p_holm"] = float(a)
    S["stats"][ds] = res

# Friedman / Nemenyi / Wilcoxon over dataset-seed blocks
blocks, meths = [], [n for n in MAIN]
for ds in S["runs"]:
    for s in range(3):
        row = []
        for n in meths:
            r = S["runs"][ds].get(n)
            if r is None or len(r["correct_seeds"]) <= s: row = None; break
            row.append(np.mean(r["correct_seeds"][s]))
        if row: blocks.append(row)
if len(blocks) >= 3:
    A = np.array(blocks)
    ranks = np.array([stats.rankdata(-r) for r in A])
    fr = stats.friedmanchisquare(*A.T)
    q = {12: 3.268, 11: 3.219, 10: 3.164}[len(meths)]  # Studentized range / sqrt(2), alpha 0.05
    cd = q * np.sqrt(len(meths) * (len(meths) + 1) / (6 * len(A)))
    wil = {}
    oi = meths.index("Ours")
    for j, n in enumerate(meths):
        if n == "Ours": continue
        try: wil[n] = float(stats.wilcoxon(A[:, oi], A[:, j]).pvalue)
        except ValueError: wil[n] = 1.0
    S["friedman"] = dict(chi2=float(fr.statistic), p=float(fr.pvalue), cd=float(cd), nblocks=len(A),
                         mean_rank={n: float(ranks[:, j].mean()) for j, n in enumerate(meths)}, wilcoxon=wil)

for ds in S["runs"]:
    for n in S["runs"][ds]:
        S["runs"][ds][n].pop("correct_seeds", None)
json.dump(S, open("results/summary.json", "w"))
print("ok", {ds: len(S["runs"][ds]) for ds in S["runs"]})


### `flops.py`

In [ ]:
%%writefile flops.py
"""Measured FLOPs (torch FlopCounterMode, multiply-add counted as 2 FLOPs) and CPU latency."""
import json, time, numpy as np, torch
from torch.utils.flop_counter import FlopCounterMode
import kairos as kz
from plans import METHODS
from common import load_encoder, probe_energy

torch.set_num_threads(1)
out = {}
enc = load_encoder()
x = torch.randn(1, 3, 112, 112)
with FlopCounterMode(display=False) as fc:
    enc.forward_features(x)
out["enc_gflops_per_frame"] = fc.get_total_flops() / 1e9
with torch.inference_mode():
    xb = torch.randn(8, 3, 112, 112); enc.forward_features(xb)
    t = time.time()
    for _ in range(5): enc.forward_features(xb)
    out["enc_ms_per_frame"] = (time.time() - t) / 40 * 1e3
# probe: measured time on 240x320 frames, analytic op count
fr = [np.random.randint(0, 255, (240, 320, 3), np.uint8) for _ in range(64)]
t = time.time(); probe_energy(fr); out["probe_ms_per_frame"] = (time.time() - t) / 64 * 1e3
out["probe_mflops_per_frame"] = (240 * 320 * 3 * 2 + 112 * 112 * 5 + 112 * 112 * 2 + 56 * 56 * 3) / 1e6
# heads
D, C, K = 192, 51, 8
heads = {}
for name in ["Uniform-CLS", "Dense-CLS", "Uniform-Full", "Uniform-STTS", "Uniform-ToMe", "Uniform-EViT", "Uniform-RLT", "Ours",
             "Sens-K4", "Sens-K16", "Sens-U-Full-K4", "Sens-U-Full-K16", "Sens-beta0.125", "Sens-beta0.5", "Sens-beta1.0"]:
    cfg = dict(METHODS[name]); k = cfg.pop("K"); k = 32 if k == "T" else k
    m = kz.build(type("B", (), dict(D=D, C=C))(), cfg).eval()
    X = torch.randn(1, k, 1 + kz.NP, D); M = torch.rand(1, k, kz.NP); tau = torch.linspace(0, 1, k)[None]
    with torch.no_grad():
        with FlopCounterMode(display=False) as fc:
            m(X, M, tau)
        m(X, M, tau); t = time.time()
        for _ in range(20): m(X, M, tau)
    heads[name] = dict(K=k, gflops=fc.get_total_flops() / 1e9, ms=(time.time() - t) / 20 * 1e3,
                       params=sum(p.numel() for p in m.parameters()))
out["heads"] = heads
json.dump(out, open("results/flops.json", "w"), indent=1)
print(json.dumps(out, indent=1))


### `fig_arch.py`

In [ ]:
%%writefile fig_arch.py
"""Fig. 1: architecture diagram (grayscale, no overlapping elements)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 7.2})
fig = plt.figure(figsize=(7.16, 3.45), dpi=300)
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 100); ax.set_ylim(-6.5, 46.5); ax.axis("off")


def box(x0, y0, x1, y1, text, fill="white", hatch=None, bold_title=None):
    p = FancyBboxPatch((x0, y0), x1 - x0, y1 - y0, boxstyle="round,pad=0.25,rounding_size=0.9",
                       fc=fill, ec="black", lw=0.8, hatch=hatch)
    ax.add_patch(p)
    if bold_title:
        ax.text((x0 + x1) / 2, y1 - 1.6, bold_title, ha="center", va="top", fontsize=7.4, fontweight="bold")
        ax.text((x0 + x1) / 2, (y0 + y1) / 2 - 1.3, text, ha="center", va="center", fontsize=6.6, linespacing=1.25)
    else:
        ax.text((x0 + x1) / 2, (y0 + y1) / 2, text, ha="center", va="center", fontsize=6.8, linespacing=1.25)
    return dict(l=x0, r=x1, b=y0, t=y1, cx=(x0 + x1) / 2, cy=(y0 + y1) / 2)


def arrow(pts, dashed=False, lw=0.9, head=True):
    for (xa, ya), (xb, yb) in zip(pts[:-2], pts[1:-1]):
        ax.plot([xa, xb], [ya, yb], color="black", lw=lw, ls=(0, (3.2, 2.0)) if dashed else "-", solid_capstyle="butt")
    a = FancyArrowPatch(pts[-2], pts[-1], arrowstyle="-|>" if head else "-", mutation_scale=7.5, lw=lw, color="black",
                        linestyle=(0, (3.2, 2.0)) if dashed else "-", shrinkA=0, shrinkB=0)
    ax.add_patch(a)


def label(x, y, s, ha="center", va="center", size=6.3, style="italic"):
    ax.text(x, y, s, ha=ha, va=va, fontsize=size, style=style,
            bbox=dict(fc="white", ec="none", pad=0.6))


# ---------------- top row: decide before encoding
A1 = box(1.0, 29.5, 12.5, 39.5, "Raw stream\n$\\{I_r\\}_{r=1}^{R}$")
A2 = box(16.5, 29.5, 35.0, 39.5, "$|g_r-g_{r-1}|$ per cell, 56$\\times$56\nspatial median: camera\ntemporal median: persistent", bold_title="Pixel motion probe")
A3 = box(39.0, 29.5, 52.5, 39.5, "$\\hat{E}_{j,n}$ on lattice\n$v_j=\\sum_n \\hat{E}_{j,n}$", bold_title="Velocity field")
A4 = box(56.5, 29.5, 75.0, 39.5, "$p_j\\propto(1-\\rho)\\,v_j^{\\gamma}/\\Sigma+\\rho/T$\nquantiles $\\to j_1<\\dots<j_K$", bold_title="Coverage-floored sampler")
A5 = box(79.0, 29.5, 99.0, 39.5, "", fill="#ececec", hatch="////")
# white patch behind the frozen-encoder text so hatching does not cross it
ax.add_patch(FancyBboxPatch((80.6, 31.0), 16.8, 7.2, boxstyle="round,pad=0.1,rounding_size=0.5", fc="#ececec", ec="none"))
ax.text(A5["cx"], 37.9, "Frozen encoder", ha="center", va="top", fontsize=7.4, fontweight="bold")
ax.text(A5["cx"], A5["cy"] - 1.3, "DINOv2-S/14 on the\n$K$ selected frames only", ha="center", va="center", fontsize=6.6, linespacing=1.25)

# ---------------- bottom row: allocate, encode geometry, reason
B1 = box(73.0, 8.0, 99.0, 18.5, "$s_{k,n}=w_m\\mathcal{Z}(\\mathrm{motion})+w_n\\mathcal{Z}(\\mathrm{novelty})+g_\\phi$\nglobal top-$B$ + mass context tokens", bold_title="Cross-frame token budget")
B2 = box(49.5, 8.0, 69.0, 18.5, "order $\\mathbf{o}_k$ + zero-init gates on\n$\\theta(\\tau_k),\\ \\theta(\\ell_k)$, velocity $\\mathbf{u}_{k,n}$", bold_title="Sampling-geometry encoding")
B3 = box(27.0, 8.0, 45.5, 18.5, "$L=2$ blocks, $d=128$\nproportional attention", bold_title="Spatiotemporal Transformer")
B4 = box(13.0, 8.0, 23.0, 18.5, "Classifier\n$\\hat{\\mathbf{y}}$")
B5 = box(1.0, 8.0, 9.0, 18.5, "Loss\n$\\mathcal{L}$")

# ---------------- forward arrows
arrow([(A1["r"], 34.5), (A2["l"] - 0.4, 34.5)])
arrow([(A2["r"], 34.5), (A3["l"] - 0.4, 34.5)])
arrow([(A3["r"], 34.5), (A4["l"] - 0.4, 34.5)])
arrow([(A4["r"], 34.5), (A5["l"] - 0.4, 34.5)]); label(77.0, 36.6, "$j_k$")
# raw frames fetched for the selected indices only (route above all boxes)
arrow([(A1["cx"], A1["t"]), (A1["cx"], 44.0), (A5["cx"], 44.0), (A5["cx"], A5["t"] + 0.4)])
label(46.0, 44.0, "decode only $I_{c_{j_k}}$ (no other frame reaches the encoder)")
# tokens down to the allocator
arrow([(92.0, A5["b"]), (92.0, B1["t"] + 0.4)]); label(95.6, 24.0, "$\\mathbf{x}_{k,n}$")
# motion energy of the selected frames down to the allocator
arrow([(71.0, A4["b"]), (71.0, 24.0), (80.0, 24.0), (80.0, B1["t"] + 0.4)]); label(75.5, 25.9, "$\\hat{E}_{j_k,n}$")
# timestamps to the geometry encoder
arrow([(60.0, A4["b"]), (60.0, B2["t"] + 0.4)]); label(56.3, 24.0, "$\\tau_k,\\ \\Delta_k$")
# bottom row, right to left
arrow([(B1["l"], 13.25), (B2["r"] + 0.4, 13.25)]); label(71.0, 20.8, "$1+2K+B$ tokens", size=6.0)
arrow([(B2["l"], 13.25), (B3["r"] + 0.4, 13.25)]); label(47.5, 15.3, "$\\mathbf{h}$")
arrow([(B3["l"], 13.25), (B4["r"] + 0.4, 13.25)]); label(25.0, 15.3, "$\\mathbf{z}_0$")
arrow([(B4["l"], 13.25), (B5["r"] + 0.4, 13.25)])

# ---------------- reverse-mode gradient (dashed), stops at the straight-through gate
gy = 3.0
arrow([(B5["cx"], B5["b"]), (B5["cx"], gy), (86.0, gy), (86.0, B1["b"] - 0.4)], dashed=True)
for xg in (18.0, 36.25, 59.25):
    arrow([(xg, gy), (xg, B2["b"] - 0.4)], dashed=True)
label(47.6, gy - 0.05, "$\\partial\\mathcal{L}/\\partial(\\theta,\\phi,w_m,w_n,\\alpha_\\tau,\\alpha_u)$ through the straight-through gate $1+\\sigma(s)-\\mathrm{sg}[\\sigma(s)]$", size=6.2)
# stop-gradient marks
for (x, y) in [(A5["cx"] - 7.0, A5["b"] - 2.0), (A4["cx"] - 3.5, A4["b"] - 2.0)]:
    pass
ax.text(A5["l"] + 1.0, A5["b"] - 1.9, "no gradient", ha="left", va="center", fontsize=6.0, style="italic")
ax.text(59.0, A4["b"] - 1.9, "non-differentiable", ha="right", va="center", fontsize=6.0, style="italic")

# ---------------- legend row (separate from all content)
ly = -3.6
ax.plot([3, 9], [ly, ly], color="black", lw=0.9); ax.text(10, ly, "forward data flow", va="center", fontsize=6.4)
ax.plot([30, 36], [ly, ly], color="black", lw=0.9, ls=(0, (3.2, 2.0))); ax.text(37, ly, "reverse-mode gradient", va="center", fontsize=6.4)
ax.add_patch(FancyBboxPatch((60, ly - 1.2), 5, 2.4, boxstyle="round,pad=0.1,rounding_size=0.3", fc="#ececec", ec="black", lw=0.6, hatch="////"))
ax.text(66.5, ly, "frozen, never updated", va="center", fontsize=6.4)
ax.text(99, ly, "upper row: decided before encoding", va="center", ha="right", fontsize=6.4, style="italic")

fig.savefig("figs/fig1_architecture.png", dpi=300)
print("ok")


### `fig_sampler.py`

In [ ]:
%%writefile fig_sampler.py
"""Fig. 2: probe velocity, compensation and the three samplers on one TRACE-10 video (grayscale)."""
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from synth import render, R
from common import probe_energy, lattice, aggregate
import kairos as kz

plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 8})
# pick a video with a camera bump well separated from a fast event
for vid in range(20000, 20400):
    fr, m = render(vid, 0)
    if m["shake"] > 0 and abs(m["shake"] - m["t0"]) > 50: break
e, res = probe_energy(fr)
T, K = 48, 8
idx = lattice(R, T)
Eraw, Eres = aggregate(e, idx), aggregate(res, idx)
vraw, vres = Eraw.sum(1), Eres.sum(1)
samp = {"Uniform": kz.uniform_sample(T, K), "MGSampler": kz.quantile_sample(vraw, K, 0.0, 1.0),
        r"KAIROS ($\rho=0.2$)": kz.quantile_sample(kz.tmed(Eres).sum(1), K, 0.2, 2.0)}

fig = plt.figure(figsize=(3.5, 4.25), dpi=300)
gs = fig.add_gridspec(3, 1, height_ratios=[1.15, 1.0, 0.62], hspace=0.62, left=0.13, right=0.97, top=0.97, bottom=0.04)
ax = fig.add_subplot(gs[0])
r = np.arange(R)
ax.plot(r, e.sum(1) / e.sum(1).max(), color="0.6", lw=0.9, label="raw difference energy")
ax.plot(r, res.sum(1) / e.sum(1).max(), color="black", lw=1.0, label="camera-compensated residual")
ax.axvspan(m["t0"], m["t1"], color="0.85", lw=0)
ax.text((m["t0"] + m["t1"]) / 2, 1.08, "event", ha="center", va="bottom", fontsize=7)
ax.annotate("camera bump", xy=(m["shake"] + 3, 0.92), xytext=(m["shake"] + 3 + (30 if m["shake"] < 120 else -60), 0.62),
            fontsize=7, arrowprops=dict(arrowstyle="-|>", lw=0.6, color="black"))
ax.set_xlim(0, R - 1); ax.set_ylim(0, 1.25); ax.set_xlabel("raw frame index $r$"); ax.set_ylabel("normalized energy")
ax.legend(loc="upper left", fontsize=6.5, frameon=False, bbox_to_anchor=(0.0, 1.0), handlelength=1.6)
ax.set_title("(a) probe output", fontsize=8, loc="left", pad=3)

ax = fig.add_subplot(gs[1])
vt = kz.tmed(Eres).sum(1) ** 2; p = 0.8 * vt / vt.sum() + 0.2 / T
ax.step(np.arange(T), np.cumsum(p), where="post", color="black", lw=1.0, label=r"$F_j$ (KAIROS)")
ax.step(np.arange(T), np.cumsum(vraw / vraw.sum()), where="post", color="0.55", lw=0.9, ls="--", label="cumulative raw motion")
ev = np.where((idx >= m["t0"]) & (idx < m["t1"] + 4))[0]
ax.axvspan(ev.min() - 0.5, ev.max() + 0.5, color="0.85", lw=0)
for q in (np.arange(K) + .5) / K:
    ax.axhline(q, color="0.8", lw=0.4, zorder=0)
ax.set_xlim(-0.5, T - 0.5); ax.set_ylim(0, 1.02); ax.set_xlabel("lattice position $j$"); ax.set_ylabel("cumulative mass")
ax.legend(loc="lower right", fontsize=6.5, frameon=False)
ax.set_title("(b) cumulative distributions and quantile targets", fontsize=8, loc="left", pad=3)

ax = fig.add_subplot(gs[2])
marks = ["|", "x", "o"]
for row, ((name, s), mk) in enumerate(zip(samp.items(), marks)):
    yv = 2 - row
    ax.plot(s, [yv] * len(s), ls="none", marker=mk, color="black", ms=5 if mk != "o" else 3.6, mfc="none" if mk == "o" else None, mew=0.9)
    hits = int(((idx[s] >= m["t0"]) & (idx[s] < m["t1"] + 4)).sum())
    ax.text(T + 0.6, yv, f"{hits}/8", va="center", fontsize=7)
ax.axvspan(ev.min() - 0.5, ev.max() + 0.5, color="0.85", lw=0)
ax.set_yticks([2, 1, 0]); ax.set_yticklabels(list(samp.keys()), fontsize=7)
ax.set_xlim(-0.5, T + 4.5); ax.set_ylim(-0.6, 2.6); ax.set_xlabel("lattice position $j$ (right: samples inside the event)")
ax.set_title("(c) selected frames", fontsize=8, loc="left", pad=3)
for a in fig.axes:
    for sp in ("top", "right"): a.spines[sp].set_visible(False)
fig.savefig("figs/fig2_sampler.png", dpi=300)
print(vid, m, {k: v.tolist() for k, v in samp.items()})


### `fig_tokens.py`

In [ ]:
%%writefile fig_tokens.py
"""Fig. 3: frames and retained regions for one TRACE-10 test video (grayscale thumbnails)."""
import json, numpy as np, torch, matplotlib, cv2
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import kairos as kz
from plans import METHODS
from synth import render, CLASSES

plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 7.5})
b = kz.Bank("data/trace"); meta = json.load(open("data/trace/meta.json"))["metas"]
te = b.ids(2)
models = {}
for name in ("Uniform-STTS", "Ours"):
    cfg = dict(METHODS[name]); cfg.pop("K")
    m = kz.build(b, cfg); m.load_state_dict(torch.load(f"results/trace/{name}/seed0/model.pt", map_location="cpu"), strict=False); m.eval()
    models[name] = (m, cfg)


def kept_mask(m, cfg, i):
    idx = kz.select(b, i, 8, cfg["sampler"])
    X = torch.from_numpy(b.tok[i, idx].astype(np.float32))[None]; M = torch.from_numpy(b.Etm[i, idx])[None]
    P = X[:, :, 1:]; Prev = torch.cat([P[:, :1], P[:, :-1]], 1); cls_t = X[:, :, 0]
    with torch.no_grad():
        s, _ = m._scores(P, Prev, M, cls_t, cfg["reducer"])
        if cfg["reducer"] == "ours":
            top = s.reshape(1, -1).topk(cfg["B"], 1).indices
            keep = torch.zeros(1, 8 * kz.NP, dtype=torch.bool).scatter_(1, top, True).view(8, kz.NP)
        else:
            top = s.topk(cfg["B"] // 8, -1).indices
            keep = torch.zeros(1, 8, kz.NP, dtype=torch.bool).scatter_(2, top, True)[0]
        pred = m(X, M, torch.from_numpy(idx / (b.T - 1)).float()[None]).argmax(1).item()
    return idx, keep.numpy(), pred


# choose a test video of a fast class where KAIROS is correct and the uniform model is wrong
pick = None
for i in te:
    if b.y[i] in (0, 2, 4) and meta[i]["shake"] < 0:
        a = kept_mask(*models["Ours"], i); u = kept_mask(*models["Uniform-STTS"], i)
        if a[2] == b.y[i] and u[2] != b.y[i]:
            pick = i; break
i = pick if pick is not None else te[0]
fr, mt = render(int(i), int(b.y[i]))
lat = np.array(meta[i]["lattice"])
fig, axs = plt.subplots(2, 8, figsize=(7.16, 2.35), dpi=300)
fig.subplots_adjust(left=0.085, right=0.995, top=0.86, bottom=0.04, wspace=0.06, hspace=0.32)
info = {}
for row, name in enumerate(("Uniform-STTS", "Ours")):
    idx, keep, pred = kept_mask(*models[name], i)
    info[name] = dict(frames=lat[idx].tolist(), kept_per_frame=keep.sum(1).tolist(), pred=CLASSES[pred])
    for k in range(8):
        ax = axs[row, k]; r = int(lat[idx[k]])
        g = cv2.cvtColor(fr[r], cv2.COLOR_RGB2GRAY).astype(np.float32)
        img = np.stack([g] * 3, -1) / 255.0
        for n in range(kz.NP):
            y0, x0 = (n // 4) * 28, (n % 4) * 28
            if not keep[k, n]:
                img[y0:y0 + 28, x0:x0 + 28] = 0.55 * img[y0:y0 + 28, x0:x0 + 28] + 0.45
        ax.imshow(img, interpolation="nearest")
        for n in range(kz.NP):
            if keep[k, n]:
                y0, x0 = (n // 4) * 28, (n % 4) * 28
                ax.add_patch(Rectangle((x0 + 0.5, y0 + 0.5), 27, 27, fill=False, ec="black", lw=0.9))
        inside = mt["t0"] <= r < mt["t1"] + 4
        ax.set_title(f"$r$={r}" + (" (event)" if inside else ""), fontsize=6.5, pad=2)
        ax.set_xticks([]); ax.set_yticks([])
    axs[row, 0].set_ylabel("STTS-style\nuniform" if row == 0 else "KAIROS", fontsize=7.5)
fig.text(0.5, 0.965, f"true class: {CLASSES[b.y[i]]};  STTS-style predicts {info['Uniform-STTS']['pred']},  KAIROS predicts {info['Ours']['pred']}",
         ha="center", va="center", fontsize=7.5)
fig.savefig("figs/fig3_tokens.png", dpi=300)
json.dump(dict(video=int(i), event=[mt["t0"], mt["t1"]], **info), open("results/fig3_info.json", "w"), indent=1)
print(info)


### `figs_results.py`

In [ ]:
%%writefile figs_results.py
"""Result figures (grayscale, markers/hatches carry identity, legends kept clear of data)."""
import json, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 8, "axes.linewidth": 0.7,
                     "xtick.major.width": 0.6, "ytick.major.width": 0.6})
S = json.load(open("results/summary.json")); FL = json.load(open("results/flops.json"))
DSN = {"trace": "TRACE-10", "hmdb": "HMDB51", "ssv2": "SSv2 subset"}
LAB = {"TSN-mean": "TSN mean", "Uniform-CLS": "Uniform-CLS", "Dense-CLS": "Dense-CLS", "Uniform-Full": "Uniform-Full",
       "MG-Full": "MG-Full", "Uniform-Random": "Random drop", "Uniform-STTS": "STTS-style", "Uniform-ToMe": "ToMe",
       "Uniform-EViT": "EViT", "Uniform-RLT": "RLT-style", "MG-STTS": "MG + STTS", "Ours": "KAIROS"}
MK = {"TSN-mean": "v", "Uniform-CLS": "^", "Dense-CLS": "D", "Uniform-Full": "s", "MG-Full": "P", "Uniform-Random": "x",
      "Uniform-STTS": "o", "Uniform-ToMe": "<", "Uniform-EViT": ">", "Uniform-RLT": "h", "MG-STTS": "X", "Ours": "*"}


def clean(ax):
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)


def gflops(ds, name):
    """Encoder + probe + temporal model GFLOPs per video."""
    d = S["data"][ds]; enc = FL["enc_gflops_per_frame"]; h = FL["heads"]
    K = d["T"] if name == "Dense-CLS" else 8
    head = {"TSN-mean": 0.0}.get(name)
    if head is None:
        key = name if name in h else ("Uniform-STTS" if name in ("Uniform-Random", "MG-STTS") else
                                      "Uniform-Full" if name == "MG-Full" else "Uniform-CLS")
        head = h[key]["gflops"] if name != "Dense-CLS" else h["Dense-CLS"]["gflops"] * (d["T"] / h["Dense-CLS"]["K"])
    probe = d["nraw_mean"] * FL["probe_mflops_per_frame"] / 1e3 if name in ("MG-Full", "MG-STTS", "Ours") else 0.0
    return K * enc + head + probe


def fig_pareto():
    """Dot plot of top-1 accuracy per method and dataset (mean ± sd over seeds); cost is given in Table main."""
    order = list(LAB)[::-1]
    fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.75), dpi=300, sharey=True)
    fig.subplots_adjust(left=0.165, right=0.99, top=0.9, bottom=0.15, wspace=0.12)
    for ax, ds in zip(axs, ["trace", "hmdb", "ssv2"]):
        R = S["runs"][ds]
        for yi, n in enumerate(order):
            m, s_, _ = R[n]["test"]
            ax.errorbar(100 * m, yi, xerr=100 * s_, fmt=MK[n], color="black", mfc="black" if n == "Ours" else "white",
                        ms=7.5 if n == "Ours" else 4.2, mew=0.8, elinewidth=0.7, capsize=1.6)
        best = max(100 * R[n]["test"][0] for n in order if n != "Ours")
        ax.axvline(best, color="0.6", lw=0.6, ls=":")
        ax.set_title(DSN[ds], fontsize=8.5); ax.set_xlabel("top-1 accuracy (%)")
        ax.grid(True, axis="x", color="0.9", lw=0.5); clean(ax)
    axs[0].set_yticks(range(len(order))); axs[0].set_yticklabels([LAB[n] + (" (6× enc.)" if n == "Dense-CLS" else "") for n in order], fontsize=7)
    axs[0].set_ylim(-0.7, len(order) - 0.3)
    fig.savefig("figs/fig4_pareto.png", dpi=300); plt.close(fig)


def fig_ablation():
    cfgs = [f"Abl-A{a}B{b}C{c}" for a in (0, 1) for b in (0, 1) for c in (0, 1)]
    labels = ["base", "+C", "+B", "+B+C", "+A", "+A+C", "+A+B", "full"]
    fig, ax = plt.subplots(figsize=(3.5, 2.35), dpi=300); fig.subplots_adjust(left=0.15, right=0.99, top=0.84, bottom=0.17)
    hatch = {"trace": "", "hmdb": "////", "ssv2": ""}; face = {"trace": "white", "hmdb": "white", "ssv2": "0.35"}
    w = 0.26
    for i, ds in enumerate(["trace", "hmdb", "ssv2"]):
        R = S["runs"][ds]; base = R["Abl-A0B0C0"]["test"][0]
        vals = [(R["Ours" if c == "Abl-A1B1C1" else c]["test"]) for c in cfgs]
        ax.bar(np.arange(8) + (i - 1) * w, [100 * (v[0] - base) for v in vals], w, yerr=[100 * v[1] for v in vals],
               color=face[ds], edgecolor="black", hatch=hatch[ds], lw=0.6, error_kw=dict(lw=0.6, capsize=1.2), label=DSN[ds])
    ax.axhline(0, color="black", lw=0.6); ax.set_xticks(range(8)); ax.set_xticklabels(labels, fontsize=7)
    ax.set_ylabel(r"$\Delta$ accuracy vs. base (pp)"); clean(ax); ax.grid(True, axis="y", color="0.9", lw=0.5); ax.set_axisbelow(True)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=7, frameon=False)
    fig.savefig("figs/fig5_ablation.png", dpi=300); plt.close(fig)


def fig_sensitivity():
    fig, axs = plt.subplots(1, 4, figsize=(7.16, 1.95), dpi=300)
    fig.subplots_adjust(left=0.06, right=0.99, top=0.8, bottom=0.25, wspace=0.42)
    st = {"trace": dict(ls="-", marker="o", mfc="black"), "hmdb": dict(ls="--", marker="s", mfc="white")}

    def get(ds, n):
        R = S["runs"][ds]
        if n == "Ours": return R["Ours"]["test"][0] * 100
        return R[n]["test"][0] * 100 if n in R else np.nan
    panels = [("$\\rho$", [0.0, 0.05, 0.1, 0.2, 0.4, 0.7, 1.0], lambda v: "Ours-rho0" if v == 0 else ("Ours" if v == 0.2 else f"Sens-rho{v}")),
              ("$\\gamma$", [1.0, 2.0, 3.0], lambda v: "Ours" if v == 2.0 else f"Sens-gamma{v}"),
              ("$\\beta$", [0.125, 0.25, 0.5, 1.0], lambda v: "Ours" if v == 0.25 else f"Sens-beta{v}"),
              ("$K$", [4, 8, 16], lambda v: "Ours" if v == 8 else f"Sens-K{v}")]
    for ax, (nm, xs, f) in zip(axs, panels):
        for ds in ("trace", "hmdb"):
            ax.plot(xs, [get(ds, f(x)) for x in xs], color="black", lw=0.9, ms=3.8, mew=0.7, **st[ds])
        if nm == "$\\beta$":
            for ds in ("trace", "hmdb"):
                ys = [get(ds, "Uniform-STTS" if x == 0.25 else f"Sens-STTS-beta{x}") for x in xs]
                ax.plot(xs, ys, color="0.55", lw=0.8, ms=3.2, ls=st[ds]["ls"], marker="^", mfc="white")
            ax.set_xscale("log", base=2); ax.set_xticks(xs); ax.set_xticklabels(["1/8", "1/4", "1/2", "1"])
        if nm == "$K$":
            for ds in ("trace", "hmdb"):
                ys = [get(ds, "Uniform-Full" if x == 8 else f"Sens-U-Full-K{x}") for x in xs]
                ax.plot(xs, ys, color="0.55", lw=0.8, ms=3.2, ls=st[ds]["ls"], marker="D", mfc="white")
            ax.set_xscale("log", base=2); ax.set_xticks(xs); ax.set_xticklabels(["4", "8", "16"])
        ax.set_xlabel(nm); clean(ax); ax.grid(True, color="0.9", lw=0.5)
    axs[0].set_ylabel("top-1 (%)")
    hs = [plt.Line2D([], [], color="black", lw=0.9, ms=3.8, **st["trace"]), plt.Line2D([], [], color="black", lw=0.9, ms=3.8, **st["hmdb"]),
          plt.Line2D([], [], color="0.55", lw=0.8, marker="^", mfc="white", ms=3.4), plt.Line2D([], [], color="0.55", lw=0.8, marker="D", mfc="white", ms=3.4)]
    fig.legend(hs, ["KAIROS, TRACE-10", "KAIROS, HMDB51", "STTS-style at same $\\beta$", "Uniform-Full at same $K$"],
               loc="upper center", ncol=4, fontsize=7, frameon=False, bbox_to_anchor=(0.5, 1.0))
    fig.savefig("figs/fig6_sensitivity.png", dpi=300); plt.close(fig)


def fig_robust():
    meths = ["Uniform-CLS", "Uniform-Full", "MG-Full", "Uniform-STTS", "MG-STTS", "Ours"]
    fig, axs = plt.subplots(1, 3, figsize=(7.16, 2.2), dpi=300)
    fig.subplots_adjust(left=0.065, right=0.99, top=0.8, bottom=0.2, wspace=0.3)
    phis = [1.0, 0.5, 0.25, 0.125]
    for ax, ds in zip(axs[:2], ["hmdb", "ssv2"]):
        R = S["runs"][ds]
        for n in meths:
            ys = [100 * (R[n]["test"][0] if p == 1.0 else R[n][f"dil{p}"][0]) for p in phis]
            ax.plot(range(4), ys, color="black", lw=1.2 if n == "Ours" else 0.8, marker=MK[n], ms=7 if n == "Ours" else 4,
                    mfc="black" if n == "Ours" else "white", mew=0.7, ls="-" if n in ("Ours", "Uniform-Full") else ":")
        ax.set_xticks(range(4)); ax.set_xticklabels(["1", "1/2", "1/4", "1/8"]); ax.set_xlabel("action fraction $\\varphi$ of the stream")
        ax.set_title(f"({'ab'[['hmdb','ssv2'].index(ds)]}) idle-stream dilution, {DSN[ds]}", fontsize=8); clean(ax); ax.grid(True, color="0.9", lw=0.5)
    axs[0].set_ylabel("top-1 (%)")
    ax = axs[2]
    for ds, ls in (("trace", "-"), ("hmdb", "--"), ("ssv2", ":")):
        R = S["runs"][ds]
        for n, mk in (("Uniform-Full", "s"), ("Ours", "*")):
            ys = [100 * R[n][f"K{k}"][0] if k != 8 else 100 * R[n]["test"][0] for k in (4, 8, 12, 16)]
            ax.plot([4, 8, 12, 16], ys, color="black", ls=ls, marker=mk, ms=6 if mk == "*" else 3.6, mfc="black" if mk == "*" else "white", lw=0.8, mew=0.7)
    ax.set_xticks([4, 8, 12, 16]); ax.set_xlabel("frames at test time (trained with $K=8$)"); ax.set_title("(c) frame-count transfer", fontsize=8)
    clean(ax); ax.grid(True, color="0.9", lw=0.5)
    hs = [plt.Line2D([], [], color="black", ls="-" if n in ("Ours", "Uniform-Full") else ":", marker=MK[n], mfc="black" if n == "Ours" else "white",
                     ms=6 if n == "Ours" else 4, lw=0.8) for n in meths]
    hs2 = [plt.Line2D([], [], color="black", ls=l, lw=0.8) for l in ("-", "--", ":")]
    fig.legend(hs + hs2, [LAB[n] for n in meths] + ["(c) TRACE-10", "(c) HMDB51", "(c) SSv2"], loc="upper center", ncol=9, fontsize=6.6,
               frameon=False, bbox_to_anchor=(0.5, 1.0), columnspacing=0.9, handletextpad=0.3, handlelength=1.8)
    fig.savefig("figs/fig7_robustness.png", dpi=300); plt.close(fig)


def fig_cd():
    F = S["friedman"]; mr = F["mean_rank"]; cd = F["cd"]
    names = sorted(mr, key=lambda n: mr[n]); M = len(names)
    fig, ax = plt.subplots(figsize=(3.5, 1.45), dpi=300); fig.subplots_adjust(left=0.02, right=0.98, top=0.97, bottom=0.03)
    ax.set_xlim(1 - 0.3, M + 0.3); ax.set_ylim(-0.55 - 0.42 * ((M + 1) // 2 - 1) - 0.35, 1.75); ax.axis("off")
    ax.plot([1, M], [0, 0], color="black", lw=0.8)
    for t in range(1, M + 1):
        ax.plot([t, t], [0, 0.18], color="black", lw=0.6); ax.text(t, 0.3, str(t), ha="center", va="bottom", fontsize=6.5)
    ax.plot([1, 1 + cd], [1.25, 1.25], color="black", lw=1.4); ax.text(1 + cd / 2, 1.35, f"CD = {cd:.2f}", ha="center", va="bottom", fontsize=6.5)
    half = (M + 1) // 2
    for i, n in enumerate(names):
        r = mr[n]
        if i < half:
            y = -0.55 - 0.42 * i; ax.plot([r, r, 0.75], [0, y, y], color="black", lw=0.5); ax.text(0.72, y, f"{LAB[n]} ({r:.1f})", ha="right", va="center", fontsize=6.3)
        else:
            j = M - 1 - i; y = -0.55 - 0.42 * j; ax.plot([r, r, M + 0.25], [0, y, y], color="black", lw=0.5); ax.text(M + 0.28, y, f"({r:.1f}) {LAB[n]}", ha="left", va="center", fontsize=6.3)
    # cliques: groups not significantly different from the best
    best = mr[names[0]]
    grp = [n for n in names if mr[n] - best < cd]
    if len(grp) > 1:
        ax.plot([best - 0.05, mr[grp[-1]] + 0.05], [-0.25, -0.25], color="black", lw=2.0)
    ax.set_xlim(-3.8, M + 4.0)
    fig.savefig("figs/fig8_cd.png", dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig)


def fig_perclass():
    import json as js
    classes = ["brush_hair", "cartwheel", "catch", "chew", "clap", "climb", "climb_stairs", "dive", "draw_sword", "dribble",
               "drink", "eat", "fall_floor", "fencing", "flic_flac", "golf", "handstand", "hit", "hug", "jump", "kick",
               "kick_ball", "kiss", "laugh", "pick", "pour", "pullup", "punch", "push", "pushup", "ride_bike", "ride_horse",
               "run", "shake_hands", "shoot_ball", "shoot_bow", "shoot_gun", "sit", "situp", "smile", "smoke", "somersault",
               "stand", "swing_baseball", "sword", "sword_exercise", "talk", "throw", "turn", "walk", "wave"]
    y = np.load("data/hmdb/labels.npy")[np.load("data/hmdb/split.npy") == 2]
    a = np.array(S["runs"]["hmdb"]["Ours"]["correct_mean"]); b = np.array(S["runs"]["hmdb"]["Uniform-Full"]["correct_mean"])
    d = np.array([100 * (a[y == c].mean() - b[y == c].mean()) for c in range(51)]); o = np.argsort(d)
    fig = plt.figure(figsize=(7.16, 2.6), dpi=300)
    ax = fig.add_axes([0.075, 0.37, 0.585, 0.55])
    ax.bar(range(51), d[o], color=["0.3" if v < 0 else "white" for v in d[o]], edgecolor="black", lw=0.5)
    ax.axhline(0, color="black", lw=0.6); ax.set_xticks(range(51)); ax.set_xticklabels([classes[i].replace("_", " ") for i in o], rotation=90, fontsize=5.2)
    ax.set_ylabel("KAIROS − Uniform-Full (pp)", fontsize=7); ax.set_title("(a) HMDB51 per-class difference (mean over seeds)", fontsize=8, loc="left"); clean(ax)
    ax2 = fig.add_axes([0.74, 0.37, 0.25, 0.55])
    tc = ["R", "L", "U", "D", "CW", "CCW", "grow", "shrink", "R slow", "L slow"]
    w = 0.27
    for i, (n, f, h) in enumerate([("Uniform-Full", "white", ""), ("MG-STTS", "white", "////"), ("Ours", "0.25", "")]):
        ax2.bar(np.arange(10) + (i - 1) * w, [100 * v for v in S["runs"]["trace"][n]["per_class"]], w, color=f, hatch=h, edgecolor="black", lw=0.5, label=LAB[n])
    ax2.set_xticks(range(10)); ax2.set_xticklabels(tc, rotation=90, fontsize=6.2); ax2.set_ylabel("top-1 (%)")
    ax2.set_title("(b) TRACE-10 per class", fontsize=8, loc="left"); clean(ax2)
    ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.33), ncol=3, fontsize=6.2, frameon=False, columnspacing=0.8, handlelength=1.2)
    fig.savefig("figs/fig9_perclass.png", dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig)
    return {classes[i]: float(d[i]) for i in o}


if __name__ == "__main__":
    fig_pareto(); fig_ablation(); fig_sensitivity(); fig_robust(); fig_cd(); pc = fig_perclass()
    json.dump(pc, open("results/perclass_hmdb.json", "w"))
    print("figures done")


### `fig_extra.py`

In [ ]:
%%writefile fig_extra.py
"""Fig.: burst hit rate vs event duration (TRACE-10 test) and analytic cost scaling with the number of frames."""
import json, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import kairos as kz

plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 8})
b = kz.Bank("data/trace", light=True); meta = json.load(open("data/trace/meta.json"))["metas"]; te = b.ids(2)
SM = [("Uniform", dict(name="uniform"), "s", "white"), ("MGSampler", dict(name="mg"), "^", "white"),
      ("KAIROS", dict(name="vqs"), "o", "black")]
dur = np.array([meta[i]["t1"] - meta[i]["t0"] for i in te])
out = {}
fig, axs = plt.subplots(1, 2, figsize=(3.5, 1.9), dpi=300, gridspec_kw=dict(width_ratios=[1, 1]))
fig.subplots_adjust(left=0.14, right=0.98, top=0.78, bottom=0.25, wspace=0.42)
groups = [("fast", dur < 30), ("slow", dur >= 30)]
for name, smp, mk, fc in SM:
    H, two = [], []
    for i in te:
        m = meta[i]; lat = np.array(m["lattice"]); r = lat[kz.select(b, i, 8, smp)]
        ins = (r >= m["t0"]) & (r < m["t1"] + 4); H.append(ins.mean()); two.append(ins.sum() >= 2)
    H, two = np.array(H), np.array(two)
    out[name] = {g: dict(H=float(H[msk].mean()), two=float(two[msk].mean())) for g, msk in groups}
    for ax, key in zip(axs, ["H", "two"]):
        ys = [out[name][g][key] * (100 if key == "two" else 1) for g, _ in groups]
        ax.plot([0, 1], ys, marker=mk, mfc=fc, color="black", lw=0.9, ms=4.5, mew=0.8, label=name)
for ax, yl, tt in zip(axs, ["hit rate $H$", "videos with ≥2 hits (%)"], ["(a) share of frames in event", "(b) ≥2 frames in event"]):
    ax.set_xticks([0, 1]); ax.set_xticklabels(["fast\n(10–16 fr.)", "slow\n(64–96 fr.)"], fontsize=7)
    ax.set_xlim(-0.35, 1.35); ax.set_ylabel(yl); ax.set_title(tt, fontsize=7.5, loc="left")
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.grid(True, axis="y", color="0.9", lw=0.5)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc="upper center", ncol=3, fontsize=7, frameon=False, bbox_to_anchor=(0.55, 1.0))
fig.savefig("figs/fig_hitdur.png", dpi=300); plt.close(fig)

# analytic cost scaling (FLOPs = 2 x multiply-accumulates)
FL = json.load(open("results/flops.json")); enc = FL["enc_gflops_per_frame"]
Ks = np.array([8, 16, 32, 64, 128]); L, d, N = 12, 768, 16
def head(M): return 2 * L * (12 * M * d ** 2 + 2 * M ** 2 * d) / 1e9
full = head(1 + Ks * (1 + N)); bud = head(1 + 2 * Ks + (0.25 * Ks * N - Ks)); encc = Ks * enc
fig, ax = plt.subplots(figsize=(3.5, 1.95), dpi=300); fig.subplots_adjust(left=0.15, right=0.98, top=0.8, bottom=0.22)
ax.plot(Ks, encc, color="black", lw=0.9, marker="D", mfc="white", ms=4, label="frozen encoder ($K$ passes)")
ax.plot(Ks, full, color="black", lw=0.9, ls="--", marker="s", mfc="white", ms=4, label="joint attention, all tokens")
ax.plot(Ks, bud, color="black", lw=1.1, marker="o", ms=4, label=r"clip-level budget, $\beta=1/4$")
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xticks(Ks); ax.set_xticklabels([str(k) for k in Ks])
ax.set_xlabel("frames $K$"); ax.set_ylabel("GFLOPs per video")
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
ax.grid(True, color="0.9", lw=0.5)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, fontsize=6.6, frameon=False, columnspacing=1.0)
fig.savefig("figs/fig_scaling.png", dpi=300); plt.close(fig)
out["scaling"] = dict(K=Ks.tolist(), enc=encc.tolist(), full=full.tolist(), budget=bud.tolist())
json.dump(out, open("results/extra.json", "w"), indent=1)
print(json.dumps(out, indent=1)[:900])


### `merge_chunks.py`

In [ ]:
%%writefile merge_chunks.py
import sys, glob, json, numpy as np
out = f"data/{sys.argv[1]}"; T = int(sys.argv[2])
ch = sorted(glob.glob(f"{out}/chunk_*.npz"), key=lambda f: int(np.load(f)["lo"]))
A = {k: np.concatenate([np.load(f)[k] for f in ch]) for k in ["Eres", "Eraw", "labels", "split", "nraw"]}
for k, v in A.items(): np.save(f"{out}/{k}.npy", v)
metas = sum([json.loads(str(np.load(f)["metas"])) for f in ch], [])
json.dump(dict(T=T, N=len(A["labels"]), pca_evr=json.load(open(f"{out}/pca_evr.json"))["evr"], metas=metas), open(f"{out}/meta.json", "w"))
print("merged", len(A["labels"]))


## 1. Data
HMDB51 split 1 and the SSv2 evaluation subset are distributed as parquet files on the Hugging Face Hub (`mteb/HMDB51`, `mteb/SomethingSomethingV2`). TRACE-10 is generated by `synth.py`.

In [ ]:
%%bash
mkdir -p raw/hmdb raw/ssv2 figs results
B=https://huggingface.co/datasets
for f in train-00000-of-00003 train-00001-of-00003 train-00002-of-00003 test-00000-of-00001; do
  [ -f raw/hmdb/$f.parquet ] || curl -sL -o raw/hmdb/$f.parquet $B/mteb/HMDB51/resolve/main/data/$f.parquet
done
[ -f raw/ssv2/test-00000-of-00001.parquet ] || curl -sL -o raw/ssv2/test-00000-of-00001.parquet $B/mteb/SomethingSomethingV2/resolve/main/data/test-00000-of-00001.parquet
ls -la raw/*

## 2. Probe and frozen-encoder features
Each command decodes every clip once, runs the probe on all raw frames and encodes the candidate lattice. Tokens are reduced to 192 dimensions with a PCA fitted on training clips only.

In [ ]:
!python prep.py trace
!python prep.py hmdb
!python prep.py ssv2

## 3. Experiments
`main`: 12 methods × 3 datasets × 3 seeds with stress tests; `abl`: factorial ablation and component removals; `sens`: sensitivity sweeps. Runs are resumable: finished runs are skipped.

In [ ]:
import subprocess, sys
def run(plan):
    ps = [subprocess.Popen([sys.executable, 'run_exp.py', plan, str(w), str(N_WORKERS)]) for w in range(N_WORKERS)]
    for p in ps: p.wait()
if QUICK:
    import plans
    plans.PLANS['quick'] = lambda: plans._jobs(plans.MAIN, plans.DS, [0], extra=True)
    open('plans.py','a').write("\nPLANS['quick'] = lambda: _jobs(MAIN, DS, [0], extra=True)\n")
    run('quick')
else:
    for plan in ['main', 'abl', 'sens']:
        run(plan)

## 4. Statistics, cost and figures

In [ ]:
!python flops.py > /dev/null
!python analyze.py
!python fig_arch.py && python fig_sampler.py && python fig_tokens.py && python figs_results.py && python fig_extra.py

In [ ]:
import json
S = json.load(open('results/summary.json'))
for ds in S['runs']:
    print(f'== {ds}')
    for n, r in S['runs'][ds].items():
        if n in ['TSN-mean','Uniform-CLS','Dense-CLS','Uniform-Full','MG-Full','Uniform-Random','Uniform-STTS','Uniform-ToMe','Uniform-EViT','Uniform-RLT','MG-STTS','Ours']:
            print(f"{n:16s} top-1 {100*r['test'][0]:6.2f} ± {100*r['test'][1]:.2f}   ECE {100*r['ece'][0]:.2f}")
print(json.dumps(S.get('friedman', {}), indent=1)[:800])

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figs/*.png')):
    print(f); display(Image(f, width=900))